# Model Deployment and Utilities
## Healthcare Provider Fraud Detection

This notebook contains helper functions, model serialization, and deployment utilities.

## 1. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import pickle
import json
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

## 2. Utility Functions

In [ ]:
class FraudDetectionPipeline:
    """
    Complete pipeline for fraud detection including data processing and prediction
    """
    
    def __init__(self, model=None, scaler=None):
        self.model = model
        self.scaler = scaler
        self.feature_names = None
    
    def save_model(self, model_path):
        """
        Save trained model to disk
        
        Parameters:
        - model_path: path to save model
        """
        with open(model_path, 'wb') as f:
            pickle.dump(self.model, f)
        print(f"Model saved to {model_path}")
    
    def load_model(self, model_path):
        """
        Load trained model from disk
        
        Parameters:
        - model_path: path to load model from
        """
        with open(model_path, 'rb') as f:
            self.model = pickle.load(f)
        print(f"Model loaded from {model_path}")
    
    def save_scaler(self, scaler_path):
        """
        Save feature scaler to disk
        
        Parameters:
        - scaler_path: path to save scaler
        """
        with open(scaler_path, 'wb') as f:
            pickle.dump(self.scaler, f)
        print(f"Scaler saved to {scaler_path}")
    
    def load_scaler(self, scaler_path):
        """
        Load feature scaler from disk
        
        Parameters:
        - scaler_path: path to load scaler from
        """
        with open(scaler_path, 'rb') as f:
            self.scaler = pickle.load(f)
        print(f"Scaler loaded from {scaler_path}")
    
    def predict(self, X, return_proba=True, threshold=0.5):
        """
        Make predictions on new data
        
        Parameters:
        - X: input features (DataFrame or array)
        - return_proba: whether to return probabilities
        - threshold: decision threshold for binary classification
        
        Returns:
        - predictions: predicted labels
        - probabilities (optional): fraud probability scores
        """
        if self.model is None:
            raise ValueError("Model not loaded. Call load_model() first.")
        
        # Scale features if scaler available
        if self.scaler is not None and isinstance(X, pd.DataFrame):
            numeric_cols = X.select_dtypes(include=[np.number]).columns
            X_scaled = X.copy()
            X_scaled[numeric_cols] = self.scaler.transform(X[numeric_cols])
            X = X_scaled
        elif self.scaler is not None:
            X = self.scaler.transform(X)
        
        predictions = self.model.predict(X)
        
        if return_proba:
            probabilities = self.model.predict_proba(X)[:, 1]
            return predictions, probabilities
        else:
            return predictions

print("FraudDetectionPipeline class defined successfully!")

## 3. Data Preprocessing Functions

In [ ]:
def preprocess_new_data(benf_df, inpatient_df, outpatient_df):
    """
    Preprocess new data for prediction
    
    Parameters:
    - benf_df: Beneficiary data
    - inpatient_df: Inpatient claims data
    - outpatient_df: Outpatient claims data
    
    Returns:
    - processed_data: merged and featured engineered data
    """
    from datetime import datetime
    
    # Remove duplicates
    benf_df = benf_df.drop_duplicates()
    inpatient_df = inpatient_df.drop_duplicates()
    outpatient_df = outpatient_df.drop_duplicates()
    
    # Handle missing values
    numeric_cols_benf = benf_df.select_dtypes(include=[np.number]).columns
    benf_df[numeric_cols_benf] = benf_df[numeric_cols_benf].fillna(benf_df[numeric_cols_benf].mean())
    
    numeric_cols_inp = inpatient_df.select_dtypes(include=[np.number]).columns
    inpatient_df[numeric_cols_inp] = inpatient_df[numeric_cols_inp].fillna(inpatient_df[numeric_cols_inp].mean())
    
    numeric_cols_out = outpatient_df.select_dtypes(include=[np.number]).columns
    outpatient_df[numeric_cols_out] = outpatient_df[numeric_cols_out].fillna(outpatient_df[numeric_cols_out].mean())
    
    # Engineer features
    inpatient_features = inpatient_df.groupby('BeneID').agg({
        'ClaimAmount': ['count', 'sum', 'mean'],
        'DeductibleAmtPaid': 'sum'
    }).reset_index()
    inpatient_features.columns = ['BeneID', 'InpatientClaimsCount', 'TotalInpatientCost', 
                                   'AvgInpatientClaimAmount', 'TotalInpatientDeductible']
    
    outpatient_features = outpatient_df.groupby('BeneID').agg({
        'ClaimAmount': ['count', 'sum', 'mean'],
        'DeductibleAmtPaid': 'sum'
    }).reset_index()
    outpatient_features.columns = ['BeneID', 'OutpatientClaimsCount', 'TotalOutpatientCost',
                                    'AvgOutpatientClaimAmount', 'TotalOutpatientDeductible']
    
    # Merge all data
    combined = inpatient_features.merge(outpatient_features, on='BeneID', how='outer')
    combined = combined.fillna(0)
    final_data = benf_df.merge(combined, left_on='BeneID', right_on='BeneID', how='left')
    final_data = final_data.fillna(0)
    
    return final_data

print("preprocess_new_data function defined successfully!")

## 4. Evaluation Functions

In [ ]:
def evaluate_predictions(y_true, y_pred, y_proba=None):
    """
    Evaluate model predictions
    
    Parameters:
    - y_true: actual labels
    - y_pred: predicted labels
    - y_proba: predicted probabilities (optional)
    
    Returns:
    - metrics: dictionary of evaluation metrics
    """
    from sklearn.metrics import (
        accuracy_score, precision_score, recall_score, f1_score,
        confusion_matrix, roc_auc_score, classification_report
    )
    
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1_score': f1_score(y_true, y_pred)
    }
    
    if y_proba is not None:
        metrics['roc_auc'] = roc_auc_score(y_true, y_proba)
    
    cm = confusion_matrix(y_true, y_pred)
    metrics['confusion_matrix'] = cm.tolist()
    metrics['true_negatives'] = int(cm[0, 0])
    metrics['false_positives'] = int(cm[0, 1])
    metrics['false_negatives'] = int(cm[1, 0])
    metrics['true_positives'] = int(cm[1, 1])
    
    return metrics

print("evaluate_predictions function defined successfully!")

## 5. Threshold Optimization

In [ ]:
def find_optimal_threshold(y_true, y_proba, objective='f1'):
    """
    Find optimal prediction threshold
    
    Parameters:
    - y_true: actual labels
    - y_proba: predicted probabilities
    - objective: 'f1', 'precision', 'recall' or 'balanced'
    
    Returns:
    - best_threshold: optimal threshold value
    - best_score: score at optimal threshold
    """
    from sklearn.metrics import precision_score, recall_score, f1_score
    
    thresholds = np.arange(0.1, 0.95, 0.01)
    scores = []
    
    for threshold in thresholds:
        y_pred = (y_proba >= threshold).astype(int)
        
        if objective == 'f1':
            score = f1_score(y_true, y_pred)
        elif objective == 'precision':
            score = precision_score(y_true, y_pred, zero_division=0)
        elif objective == 'recall':
            score = recall_score(y_true, y_pred, zero_division=0)
        elif objective == 'balanced':
            precision = precision_score(y_true, y_pred, zero_division=0)
            recall = recall_score(y_true, y_pred, zero_division=0)
            score = (precision + recall) / 2
        
        scores.append(score)
    
    best_idx = np.argmax(scores)
    best_threshold = thresholds[best_idx]
    best_score = scores[best_idx]
    
    return best_threshold, best_score

print("find_optimal_threshold function defined successfully!")

## 6. Model Configuration and Metadata

In [ ]:
class ModelMetadata:
    """
    Store and manage model metadata
    """
    
    def __init__(self, model_name, version, description=""):
        self.model_name = model_name
        self.version = version
        self.description = description
        self.metrics = {}
        self.features = []
        self.training_date = pd.Timestamp.now()
    
    def add_metrics(self, metrics_dict):
        """
        Add evaluation metrics
        """
        self.metrics.update(metrics_dict)
    
    def set_features(self, feature_list):
        """
        Set feature names used by model
        """
        self.features = feature_list
    
    def save_metadata(self, filepath):
        """
        Save metadata to JSON
        """
        metadata = {
            'model_name': self.model_name,
            'version': self.version,
            'description': self.description,
            'training_date': str(self.training_date),
            'metrics': self.metrics,
            'features': self.features,
            'feature_count': len(self.features)
        }
        
        with open(filepath, 'w') as f:
            json.dump(metadata, f, indent=2)
        
        print(f"Metadata saved to {filepath}")
    
    def load_metadata(self, filepath):
        """
        Load metadata from JSON
        """
        with open(filepath, 'r') as f:
            metadata = json.load(f)
        
        self.model_name = metadata['model_name']
        self.version = metadata['version']
        self.description = metadata['description']
        self.metrics = metadata['metrics']
        self.features = metadata['features']
        
        print(f"Metadata loaded from {filepath}")
    
    def __str__(self):
        return f"""
        Model: {self.model_name}
        Version: {self.version}
        Training Date: {self.training_date}
        Features: {len(self.features)}
        Performance Metrics: {self.metrics}
        """

print("ModelMetadata class defined successfully!")

## 7. Example Usage

In [ ]:
# Example: How to use the pipeline
"""
# 1. Load trained model and scaler
pipeline = FraudDetectionPipeline()
pipeline.load_model('fraud_detection_model.pkl')
pipeline.load_scaler('feature_scaler.pkl')

# 2. Load new data
benf_new = pd.read_csv('new_beneficiary_data.csv')
inp_new = pd.read_csv('new_inpatient_data.csv')
out_new = pd.read_csv('new_outpatient_data.csv')

# 3. Preprocess data
processed_data = preprocess_new_data(benf_new, inp_new, out_new)

# 4. Make predictions
predictions, probabilities = pipeline.predict(processed_data)

# 5. Evaluate (if labels available)
if 'PotentialFraud' in processed_data.columns:
    metrics = evaluate_predictions(processed_data['PotentialFraud'], predictions, probabilities)
    print(metrics)

# 6. Find optimal threshold (if labels available)
if 'PotentialFraud' in processed_data.columns:
    threshold, score = find_optimal_threshold(
        processed_data['PotentialFraud'], probabilities, objective='f1'
    )
    print(f"Optimal threshold: {threshold}, F1-Score: {score}")
"""

print("\nExample usage patterns documented!")

## 8. Summary

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║     FRAUD DETECTION DEPLOYMENT UTILITIES SUMMARY              ║
╚════════════════════════════════════════════════════════════════╝

AVAILABLE CLASSES:
  1. FraudDetectionPipeline
     - Load/save model and scaler
     - Make predictions with proper preprocessing
     - Handle threshold-based classification

  2. ModelMetadata
     - Track model information and performance
     - Save/load metadata as JSON
     - Maintain model versioning

AVAILABLE FUNCTIONS:
  1. preprocess_new_data()
     - Handle duplicates and missing values
     - Engineer features from raw claims data
     - Merge datasets appropriately

  2. evaluate_predictions()
     - Calculate comprehensive evaluation metrics
     - Generate confusion matrix
     - Compute ROC-AUC if probabilities provided

  3. find_optimal_threshold()
     - Search for best decision threshold
     - Support multiple objectives (F1, precision, recall)
     - Optimize for business requirements

DEPLOYMENT WORKFLOW:
  1. Load model and scaler → FraudDetectionPipeline
  2. Preprocess raw data → preprocess_new_data()
  3. Make predictions → pipeline.predict()
  4. Optimize threshold → find_optimal_threshold()
  5. Evaluate results → evaluate_predictions()
  6. Store metadata → ModelMetadata

PRODUCTION CONSIDERATIONS:
  - Use saved model and scaler for consistency
  - Track model metadata and versions
  - Monitor prediction performance over time
  - Implement confidence thresholds for predictions
  - Log all predictions for audit trails
  - Regular model retraining with new data

""")